# Model Finetuning
We will finetune the model in the following steps:
- Take in the finetuned dataset
- Convert to the proper format (tensors)
- Then finetune the model based

Some imports for libaries we will use and helper classes for finetuning the model. Instead of writing our own training loop and evaluation function, we will be using the given functions from pytorch directly. 

In [1]:
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
import json # For our manually labeled finetune train data
from PIL import Image
import torchvision.transforms as T
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset
import os
import sys

base = "https://raw.githubusercontent.com/pytorch/vision/main/references/detection"
files = ["engine.py", "utils.py", "coco_utils.py", "coco_eval.py", "transforms.py"]

for f in files:
    os.system(f"curl -o finetune_helpers/{f} {base}/{f}")

sys.path.append("./finetune_helpers")
from finetune_helpers.engine import train_one_epoch
from finetune_helpers import utils

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  4063  100  4063    0     0  20818      0 --:--:-- --:--:-- --:--:-- 20835
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8388  100  8388    0     0  41089      0 --:--:-- --:--:-- --:--:-- 41117
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  8409  100  8409    0     0  47684      0 --:--:-- --:--:-- --:--:-- 47778
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  6447  100  6447    0     0  32775      0 --:--:-- --:--:-- --:--:-- 32725
  % Total    % Received % Xferd  Average Speed   Tim

# Steps 1 & 2
First we will take in the finetuned dataset (*annotations.json*) and convert it into a list of dictionaries where each element of the dictinoary is either "boxes", i.e. the location of boxes on our classified images, or "labels", i.e. 1 since we are classifying tree (1) and not tree (0). 
We do this in the __init__ function.

Since we will be working with the data when training/finetuning we create a class that inherits from Dataset. 

In [2]:
class TreeDataset(Dataset):
    def __init__(self, annotations_path):
        self.root_path = annotations_path
        with open(self.root_path + "annotations.json") as f:
            all_annotations = json.load(f)

        self.annotations = []
        for a in all_annotations:
            if (all_annotations[a] != []): # drop pictures with no trees
                self.annotations.append((a, all_annotations[a]))
        self.transform = T.ToTensor()

    def __len__(self):
        return len(self.annotations)  # how many images total

    def __getitem__(self, idx):
        # print(self.annotations)
        # print(self.annotations[1])
        image_name, boxes = self.annotations[idx]
        img = Image.open(self.root_path + "raw_images/" + image_name).convert("RGB")
        img_tensor = self.transform(img)
        
        target = {
            "boxes":  torch.tensor(boxes, dtype=torch.float32),
            "labels": torch.ones(len(boxes), dtype=torch.int64),
            "image_id": torch.tensor([idx])
        }
        
        return img_tensor, target

# Sanity Check:
dataset_full = TreeDataset("./Training Classifier/")
print(f"Total images: {len(dataset_full)}")

Total images: 25


# Step 3
Now we are modifying the pre-trained Faster R-CNN model.  

In particular, we are modifying the classification layer of our Faster R-CNN model. Instead of multiple classes like airplanes, persons, chairs, etc., we will now have **only 2 classes**: tree (1) or no tree (0)

This code is heavily inspired by the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial)

In [3]:
def get_fine_tune_ready_model():
    # load a model pre-trained on COCO
    model = fasterrcnn_resnet50_fpn(weights="DEFAULT")

    num_classes = 2  # 1 class (tree) + background

    # get number of input features for the classifier (from the conv. layers / pooling)
    in_features = model.roi_heads.box_predictor.cls_score.in_features

    # replace the pre-trained head with a new one
    model.roi_heads.box_predictor = FastRCNNPredictor(in_features, num_classes)

    # Freezing most parameters (everything but the final head/FC layer)
    for param in model.backbone.parameters():
        param.requires_grad = False
    for param in model.rpn.parameters():
        param.requires_grad = False
    
    return model

# Step 4
Now we will finetune the model. This code is almost directly taken from the [TorchVision Object Detection Finetuning Tutorial](https://docs.pytorch.org/tutorials/intermediate/torchvision_tutorial.html#torchvision-object-detection-finetuning-tutorial) 

In [4]:
# train on the accelerator (mac) or on the CPU, if an accelerator is not available
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# our dataset has two classes only - background and tree
num_classes = 2
dataset = TreeDataset('./Training Classifier/')

# define training and validation data loaders
data_loader = torch.utils.data.DataLoader(
    dataset,
    batch_size=5,
    shuffle=True,
    collate_fn=utils.collate_fn
)

# get the model using our helper function
model = get_fine_tune_ready_model()

# move model to the right device
model.to(device)

# construct an optimizer
params = [p for p in model.parameters() if p.requires_grad]
optimizer = torch.optim.SGD(
    params,
    lr=0.005,
    momentum=0.9,
    weight_decay=0.0005
)

# and a learning rate scheduler
lr_scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer,
    step_size=3,
    gamma=0.1
)

num_epochs = 5

for epoch in range(num_epochs):
    # train for one epoch, printing every 10 iterations
    train_one_epoch(model, optimizer, data_loader, device, epoch, print_freq=10)
    # update the learning rate
    lr_scheduler.step()

print("That's it!")

/Users/paulrabel/tree_detector/model/finetune_helpers/engine.py:30: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=scaler is not None):


Epoch: [0]  [0/5]  eta: 0:00:26  lr: 0.001254  loss: 3.4620 (3.4620)  loss_classifier: 0.8087 (0.8087)  loss_box_reg: 0.0897 (0.0897)  loss_objectness: 2.4465 (2.4465)  loss_rpn_box_reg: 0.1171 (0.1171)  time: 5.3454  data: 0.1597
Epoch: [0]  [4/5]  eta: 0:00:02  lr: 0.005000  loss: 5.3458 (5.4194)  loss_classifier: 0.6776 (0.6369)  loss_box_reg: 0.1826 (0.1805)  loss_objectness: 4.3590 (4.3637)  loss_rpn_box_reg: 0.2498 (0.2383)  time: 2.7456  data: 0.0948
Epoch: [0] Total time: 0:00:13 (2.7465 s / it)
Epoch: [1]  [0/5]  eta: 0:00:13  lr: 0.005000  loss: 4.6148 (4.6148)  loss_classifier: 0.7952 (0.7952)  loss_box_reg: 0.2241 (0.2241)  loss_objectness: 3.3999 (3.3999)  loss_rpn_box_reg: 0.1955 (0.1955)  time: 2.6439  data: 0.0706
Epoch: [1]  [4/5]  eta: 0:00:01  lr: 0.005000  loss: 5.1012 (5.5394)  loss_classifier: 0.7833 (0.7479)  loss_box_reg: 0.1953 (0.1937)  loss_objectness: 3.7835 (4.3545)  loss_rpn_box_reg: 0.2262 (0.2433)  time: 1.9363  data: 0.0952
Epoch: [1] Total time: 0:00:0

# Final Step:
Save the learned weights

In [ ]:
torch.save(model.state_dict(), "weights.pt")